# 🛡️ Day-2 · Notebook 03: Crash-Proof Code & Safe File Handling
**Python for AI Developers · Defensive Engineering**

Welcome to Notebook 03 of Day 2!

In software engineering, external things will inevitably go wrong:
- A student file might be missing or corrupted.
- An API key might be expired.
- An LLM endpoint might return a 429 rate limit error.

In this notebook, you and I will master **defensive engineering**:
1. The 4-stage exception lifecycle: `try`, `except`, `else`, and `finally` (visual guide!).
2. Creating custom domain exceptions (e.g., `AttendanceShortageError`).
3. Safe file handling using `with open()` context managers.
4. Reading and writing student CSV and JSON datasets with automatic cleanup.

Let's look at the exception lifecycle diagram first!


### 🛡️ The 4-Stage Defensive Exception Lifecycle

Take a look at how Python catches and handles surprises gracefully:

![Exception Lifecycle](../../images/day2_exception_handling_flow.png)

1. **`try`**: Put only the risky code here.
2. **`except`**: Catches specific errors (like `FileNotFoundError` or `ValueError`). Never let your program crash!
3. **`else`**: Runs **only** if everything in the `try` block succeeded with zero errors!
4. **`finally`**: **Always runs** no matter what (perfect for closing files or logging audit trails).

Let's test this with a student record parser:


In [ ]:
def parse_exam_score(raw_text: str):
    print(f"\nAttempting to parse: {repr(raw_text)}")
    try:
        score = float(raw_text)
        if score < 0 or score > 100:
            raise ValueError(f"Score {score} is outside allowable range [0, 100]")
    except ValueError as err:
        print(f"  ❌ [EXCEPT CAUGHT]: {err}")
        return None
    else:
        print(f"  ✅ [ELSE SUCCESS]: Valid exam score: {score:.1f}%")
        return score
    finally:
        print("  📋 [FINALLY]: Telemetry audit logged.")

# Let's test all three paths:
parse_exam_score("94.5")        # Successful path -> runs try, else, finally
parse_exam_score("125.0")       # Out of bounds -> runs try, except, finally
parse_exam_score("absent")      # Bad conversion -> runs try, except, finally


---
### 📂 Step 2: Safe File Handling with `with open()`

When working with files on disk, you should never leave file descriptors dangling open.
Python's context manager (`with open(...) as f:`) guarantees that the file is **automatically flushed and closed** when the block finishes, even if your code crashes!

![File Handling Lifecycle](../../images/day2_file_handling_lifecycle.png)

Let's write a batch of student records to a CSV file and read them back:


In [ ]:
import csv
from pathlib import Path

# Let's define our file path:
csv_path = Path("kit_students_roster.csv")

# 1. Writing to a CSV file safely:
students_data = [
    ["USN", "Name", "Branch", "CGPA"],
    ["1KT23AI001", "Sundeep", "AI & Data Science", "9.45"],
    ["1KT23AI002", "Priya", "AI & Data Science", "9.10"],
    ["1KT23AI003", "Rahul", "Computer Science", "8.75"]
]

with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(students_data)

print(f"✅ Created student roster at: {csv_path.name}")

# 2. Reading from the CSV file line-by-line:
print("\n--- Reading Student Records from CSV ---")
with open(csv_path, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"Roll: {row['USN']} | {row['Name']:<10} | CGPA: {row['CGPA']}")

# Clean up file:
if csv_path.exists():
    csv_path.unlink()
    print("\n[Cleanup]: Temporary demo CSV deleted.")


---
### 🎯 Your Turn: Mini Practice Challenge!

In the cell below:
1. Write a function called `safe_divide_marks(obtained: float, maximum: float)`.
2. Wrap the division in a `try/except` block to catch `ZeroDivisionError`.
3. If `maximum == 0`, catch the error and print `"Error: Maximum marks cannot be zero!"` and return `0.0`.
4. Test with `obtained = 45` and `maximum = 0`. Run the cell!


In [ ]:
# Write your code below:
